# The Fab-Line Game — G1: one bad knob → a dead die

The first banked artifact of the fab-line game (plan `docs/plans/fab-game.md` §§1, 6; ADR 0005).
It runs the **same wafer** through the *already-validated* diffusion → oxidation → lithography →
device back end — once **in focus**, once with a single bad knob (**defocus**) — and shows the
consequence ripple end to end:

> defocus the exposure → the aerial image loses edge sharpness (**NILS** collapses) → the gate no
> longer prints reliably → those dies leave the spec window → the wafer **yield** drops, and the
> **failure trail** names *defocus* as the cause.

A center-to-edge focus bowl makes the failure **spatial** — the edge ring dies while the centre
survives — and a litho **rework** (strip resist, re-expose at corrected focus) recovers it.
**All reuse, zero new physics:** every number traces to a `chip/` module that already passes its
triad. This notebook is a thin skin over the validated harness (ADR 0002/0005 — a figure is never
in the correctness path); the assertions live in `fab_game/tests/`.

In [ ]:
%matplotlib inline
from fab_game import run_line, wafer_yield, diagnose, DEFAULT_SPECS, Variation
from fab_game.recipe import Recipe, LithoKnobs
from fab_game.demo_fab_game import compute, print_summary

result = compute()
print_summary(result)

## The wafer maps

Green = a good die, red = a dead one. The good recipe yields a full wafer; one bad knob kills an
**edge ring**; the `NILS`-vs-radius panel shows *why* (the printability floor catches the
defocused edge dies); rework brings it back.

In [ ]:
from fab_game.plots import fab_game_figure

fig = fab_game_figure(result)
fig

## Turn the focus knob yourself

Sweep the **defocus** knob and watch the yield fall and the edge ring grow. (With `ipywidgets`
installed you get a live slider; otherwise the cell prints a static sweep so the notebook still
runs headless.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fab_game.plots import _wafer_map


def show_wafer(defocus_nm=90.0, seed=0):
    """Run the line at this defocus and draw the wafer map + the worst die's failure trail."""
    recipe = Recipe(litho=LithoKnobs(defocus_nm=float(defocus_nm)))
    wafer = run_line(recipe, seed=seed, variation=Variation(), specs=DEFAULT_SPECS, grid_n=7)
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    _wafer_map(ax, wafer, f"defocus = {defocus_nm:.0f} nm")
    plt.show()
    dead = [d for d in wafer.dies if d.verdict.failed]
    if dead:
        worst = max(dead, key=lambda d: d.radius_frac)
        print(diagnose(worst))
    else:
        print(f"yield {wafer_yield(wafer):.0%} — every die printed in spec.")


try:
    from ipywidgets import interact, FloatSlider
    interact(show_wafer, defocus_nm=FloatSlider(min=0, max=320, step=10, value=90,
                                                description="defocus (nm)"),
             seed=(0, 5))
except ImportError:
    for z in (0.0, 90.0, 200.0):
        print(f"\n--- defocus = {z:.0f} nm ---")
        show_wafer(z)

## G2 — down the boule: Scheil segregation → a V_t spread

The **first new front-of-line physics** (`chip.czochralski`, cited + triad-tested). A Czochralski
boule grown from a boron melt has a **Scheil** axial profile — `C_s(z) = N_seed·(1−z)^(k−1)` — so
because boron's segregation coefficient `k ≈ 0.8 < 1`, the solid that freezes later is *more* doped.
Slice a wafer batch down the boule and that rising substrate doping **alone** — one crystal-growth
knob, no process change — walks the device `V_t` up across the spec window, so the boule's **tail is
scrapped**. (Unit-of-run, plan §10: a *run* is one wafer at axial `slice_z`; `run_batch` is the
sweep that surfaces where each slice sits on the Scheil curve.)


In [ ]:
from fab_game.demo_boule import compute as compute_boule, print_summary as print_boule
from fab_game.plots import boule_figure

boule_result = compute_boule()
print_boule(boule_result)
boule_figure(boule_result)


## G3 — the die map made physical: killer particles → functional yield + the TTV scrap

New cited physics (`chip.wafer_prep`, triad-tested): the **defect-limited yield law**. Killer
particles are scattered at **locations** across the wafer; a die that catches one is dead
*functionally* (the transistor exists but is killed). The probability a die of critical area `A`
catches **zero** killers at density `D₀` is `Y = exp(−D₀·A)` — the **Murphy / Poisson** law.

The placement is a seeded **per-die Poisson** process; sweep the density and the empirical defect
yield **converges to the cited `exp(−D₀·A)` curve** — the random placement tied back to the
validated physics (same byte-identical die area on both sides). Geometry (TTV/bow) is a **wafer-level
scrap** gate: a weak CMP leaves the TTV out of flatness spec → the wafer is scrapped, recoverable by
a re-polish that **eats thickness**.


In [ ]:
from fab_game.demo_wafer_prep import compute as compute_prep, print_summary as print_prep
from fab_game.plots import wafer_prep_figure

prep_result = compute_prep()
print_prep(prep_result)
wafer_prep_figure(prep_result)


## Command the whole line — the guided slice

The cells above tour the line piece by piece; this one is the **guided slider-driven slice** (plan
§9): pick a recipe, run the *whole* line sand → binned chip, and watch the **wafer map** and the
**failure trail** react. Four principal knobs — chosen to be dramatic *and* legible:

| knob | what it does | the story |
|---|---|---|
| **defocus (nm)** | focus error at the gate exposure | with the center-to-edge focus bowl, kills an **edge ring** (a map-texture story) |
| **D₀ (cm⁻²)** | killer-particle density | scattered functional kills (the other map-texture story) |
| **boule z** | axial slice down the boule | Scheil segregation walks the substrate doping → **V_t** drifts up out of spec toward the tail (the difficulty curve) |
| **t_ox (min)** | gate-oxide drive | thinning the oxide lowers **V_t** — the lever that pulls a drifted tail back into spec (mind the over-thinning limit) |

Every other knob (purification grade, etch over/under-etch, the crystal-growth deepenings) keeps its
own focused demo. This is **all reuse, zero new physics** — a thin skin over `run_dashboard`, with
the within-wafer variation **on** so the map shows real spatial structure (an off-variation run would
collapse every die to one identity → a binary all-pass/all-fail flip, not a story). The load-bearing
run + render is smoke-tested in `fab_game/tests/test_dashboard.py`; the slider is sugar on top.

> Try it: slide **defocus** up to ~90 nm (the edge ring), then 200+ (a wipeout). Set **boule z** to
> ~0.85 (the tail drifts out the top) and *rescue* it by dropping **t_ox** to ~18 min — then keep
> dropping it and watch the rescue overshoot (the Goldilocks limit).

In [ ]:
from fab_game.dashboard import run_dashboard, dashboard_summary
from fab_game.plots import dashboard_figure

# The load-bearing call lives in this *direct* cell (not the interact callback below): interact runs
# its callback inside an Output context that captures exceptions, so a broken call there would paint
# an error yet leave the notebook "successful". The validated run + render is exercised here (and in
# fab_game/tests/test_dashboard.py); the slider is sugar on top.
result = run_dashboard()                       # the seam: the nominal recipe, within-wafer variation on
print(dashboard_summary(result))
dashboard_figure(result)

In [ ]:
import matplotlib.pyplot as plt


def command_the_line(defocus_nm=90.0, defect_density=0.0, slice_z=0.0, oxide_minutes=20.0, seed=0):
    """Run the whole line at these knobs, draw the wafer map, and print the failure trail."""
    result = run_dashboard(defocus_nm=defocus_nm, defect_density=defect_density,
                           slice_z=slice_z, oxide_minutes=oxide_minutes, seed=int(seed))
    dashboard_figure(result)
    plt.show()
    print(dashboard_summary(result))


# With ipywidgets installed you get live sliders; otherwise the cell prints a static tour (the
# clean baseline → a defocus ring → the boule-tail V_t drift → the gate-oxide rescue) so the
# notebook still executes headless.
try:
    from ipywidgets import FloatSlider, IntSlider, interact

    interact(
        command_the_line,
        defocus_nm=FloatSlider(min=0, max=250, step=10, value=90, description="defocus (nm)"),
        defect_density=FloatSlider(min=0.0, max=0.1, step=0.01, value=0.0, description="D₀ (cm⁻²)"),
        slice_z=FloatSlider(min=0.0, max=0.95, step=0.05, value=0.0, description="boule z"),
        oxide_minutes=FloatSlider(min=10, max=24, step=1, value=20, description="t_ox (min)"),
        seed=IntSlider(min=0, max=5, value=0, description="seed"),
    )
except ImportError:
    for kw in ({}, {"defocus_nm": 90.0}, {"slice_z": 0.85}, {"slice_z": 0.85, "oxide_minutes": 18.0}):
        print(f"\n--- {kw or 'clean baseline'} ---")
        command_the_line(**kw)

## Where this goes next

This notebook tours the front of the line (G1 mechanism → G2 boule → G3 die map) and ends on the
**guided full-line slice** above. The line itself is **complete**: sand → binned chip across G1–G7,
deepened by the crystal-growth triad (CG-1 pull-rate `k_eff`, CG-2 Voronkov `V/G`, CG-3 Stefan cap)
and the scope-edge promotions (thermal donors, under-etch, the OSF ring, interstitial dislocations →
leakage, the spike/RTA `∫D dt` budget) — each with its own cited, triad-tested `chip/` module and a
banked `fab-game-*.png`. The **named-consumer backlog is exhausted** (`docs/plans/scope-edge-backlog.md`):
every remaining physics edge lacks a device/yield consumer, which is the anti-over-build bar working
as intended.

What's left is **presentation, not physics**: a **Textual TUI** for the roguelike "command the line,
watch it ripple" feel (the dashboard above is its headless core), and a **tycoon** mode (same harness,
a different objective + scoring). Both stay deferred until wanted — the Python sim remains the
authority either way (ADR 0005).